# From PDF to Intelligence: Contextual Chunking + Multimodal Embeddings

This notebook demonstrates two advanced VoyageAI capabilities on a real PDF document:

| Model | What it does |
|-------|-------------|
| `voyage-3-lite` | Fast text embeddings (baseline) |
| `voyage-context-3` | Embeddings that understand each chunk's role in the full document |
| `voyage-multimodal-3.5` | Shared embedding space for text AND images |

**The Story:** We take the "Attention Is All You Need" paper (the Transformer paper) and show how each generation of VoyageAI models unlocks new retrieval capability — from crude keyword-like text search all the way to searching charts and diagrams with natural language.

In [ ]:
!pip install -q voyageai pymupdf pillow numpy matplotlib seaborn requests python-dotenv

In [ ]:
import os, io, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import fitz  # PyMuPDF
import requests
from PIL import Image

# ── API Key Setup (Google Colab compatible) ──────────────────────────────────
try:
    from google.colab import userdata
    VOYAGE_API_KEY = userdata.get('VOYAGE_API_KEY')
    print("Loaded API key from Google Colab Secrets")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    VOYAGE_API_KEY = os.environ.get('VOYAGE_API_KEY')
    print("Loaded API key from .env file")
except Exception:
    VOYAGE_API_KEY = None
    print("Could not load API key from Colab Secrets")

assert VOYAGE_API_KEY, "VOYAGE_API_KEY not found — set it in Colab Secrets or .env"

import voyageai
vo = voyageai.Client(api_key=VOYAGE_API_KEY)
print("VoyageAI client ready.")

In [ ]:
# ── Download PDF ─────────────────────────────────────────────────────────────
PDF_URL = "https://arxiv.org/pdf/1706.03762"
PDF_PATH = "/tmp/attention_is_all_you_need.pdf"

if not os.path.exists(PDF_PATH):
    print(f"Downloading PDF from {PDF_URL} ...")
    r = requests.get(PDF_URL, stream=True, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    r.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved to {PDF_PATH} ({os.path.getsize(PDF_PATH)//1024} KB)")
else:
    print(f"PDF already cached at {PDF_PATH}")

# Verify it's a valid PDF
doc = fitz.open(PDF_PATH)
print(f"PDF has {len(doc)} pages.")
doc.close()

In [ ]:
# ── Helper Functions ──────────────────────────────────────────────────────────

def cosine(a, b):
    """Cosine similarity between two vectors."""
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        raise ValueError("Cannot compute cosine similarity with a zero vector")
    return float(np.dot(a, b) / (na * nb))

def standard_chunk(text, size=500, overlap=50):
    """Split text into fixed-size overlapping character chunks."""
    if overlap >= size:
        raise ValueError(f"overlap ({overlap}) must be less than size ({size})")
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end])
        start += size - overlap
    return [c for c in chunks if c.strip()]

def render_page_image(pdf_path, page_num, dpi=150):
    """Render a single PDF page as a PIL Image."""
    doc = fitz.open(pdf_path)
    try:
        page = doc[page_num]
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    finally:
        doc.close()

def embed_in_batches(texts, model, input_type, batch_size=50):
    """Embed a list of texts in batches to avoid token limits."""
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        result = vo.embed(batch, model=model, input_type=input_type)
        embeddings.extend(result.embeddings)
    return embeddings

print("Helper functions defined.")